# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'N/A')}: {getattr(metadata, 'description', 'No description found')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets (@id) and fields with their @id
print("Available record sets and their fields:")
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs['@id']}  Name: {rs.get('name','')}")
    fields = rs.get('field', [])
    # fields could be dict or list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"   - Field @id: {field.get('@id', '')}  Name: {field.get('name', '')}")
        elif isinstance(field, str):
            print(f"   - Field @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, let's extract all available RecordSet @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field and a record set that has numeric data
import numpy as np

# Pick the main results record set for protein analysis (manually inspecting fields; adjust as needed):
# Example (please update to an actual relevant RecordSet and field IDs from the previous overview output!):
selected_record_set = record_set_ids[0] if record_set_ids else None

# Heuristic to choose a numeric field: pick the first column containing only numeric data
if selected_record_set:
    df = dataframes[selected_record_set]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to infer numeric columns from typical names
        numeric_fields = [col for col in df.columns if any(sub in col.lower() for sub in ['abundance', 'count', 'mw', 'weight', 'coverage', 'pi'])]
        # Try to convert to numeric
        for col in numeric_fields:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Example numeric field: {numeric_field}")

        threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 0

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())

        # Attempt groupby on a typical grouping field
        possible_groups = [col for col in df.columns if col not in numeric_fields and col.lower() in ['description', 'accession', 'sample', 'type']]
        group_field = possible_groups[0] if possible_groups else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for groupby analysis.")
    else:
        print('No numeric fields available in this record set for EDA.')
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field if available
if selected_record_set and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If there's a group field, make a boxplot
    if group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.